# 14 — BERTopic: Random 10 k Sample (en + vi)

Fits two independent BERTopic models on a random 10,000-review sample, one per language.

**Prerequisites**
```bash
uv run python src/preprocess_to_duckdb.py   # REVIEW_TEXT_PROCESSED
uv run python src/embed_to_duckdb.py        # REVIEW_EMBEDDINGS
```

## Section 1 — Imports & Config

In [5]:
import sys
sys.path.insert(0, "..")

import random
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from stopwordsiso import stopwords
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, PartOfSpeech
from bertopic.vectorizers import ClassTfidfTransformer

from src.topic_modeling import load_from_duckdb

DB_PATH     = Path("../data/hotel_reviews.db")
SAMPLE_SIZE = 10_000
RANDOM_SEED = 42
MIN_CLUSTER = 20

encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

print(f"DB          : {DB_PATH.resolve()}")
print(f"Sample size : {SAMPLE_SIZE:,}  |  seed={RANDOM_SEED}")
print(f"Encoder     : {encoder.__class__.__name__}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DB          : C:\Users\darkn\coastal-hotel\data\hotel_reviews.db
Sample size : 10,000  |  seed=42
Encoder     : SentenceTransformer


## Section 2 — Helper: Sample & Fit

In [6]:
def run_sample(language: str) -> tuple:
    """Load all reviews for *language*, draw a random 10k sample, fit BERTopic."""
    df_all, docs_all, emb_all = load_from_duckdb(db_path=DB_PATH, language=language)
    print(f"Total {language} reviews: {len(docs_all):,}")

    random.seed(RANDOM_SEED)
    idx = sorted(random.sample(range(len(docs_all)), min(SAMPLE_SIZE, len(docs_all))))

    docs       = [docs_all[i] for i in idx]
    embeddings = emb_all[idx]
    df         = df_all.iloc[idx].reset_index(drop=True)

    print(f"Sampled        : {len(docs):,}")
    print(f"Embedding shape: {embeddings.shape}")
    print(f"Sample doc [0] : {repr(docs[0][:120])}")

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
    hdbscan_model = HDBSCAN(
        min_cluster_size=MIN_CLUSTER,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=False,  # blocked by Windows AppControl
    )
    vectorizer_model = CountVectorizer(
        stop_words=list(stopwords(["vi", "en"])),
        min_df=2,
        ngram_range=(1, 2),
    )
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR":     MaximalMarginalRelevance(diversity=0.3),
        "POS":     PartOfSpeech("en_core_web_sm"),
    }

    topic_model = BERTopic(
        embedding_model=encoder,       # needed by KeyBERTInspired; won't re-encode
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ClassTfidfTransformer(),
        representation_model=representation_model,
        nr_topics="auto",
        min_topic_size=MIN_CLUSTER,
        calculate_probabilities=False,
        verbose=True,
    )
    topics, _ = topic_model.fit_transform(docs, embeddings)

    n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
    n_outlier = sum(1 for t in topics if t == -1)
    print(f"Topics  : {n_topics}")
    print(f"Outliers: {n_outlier:,} / {len(topics):,} ({n_outlier/len(topics)*100:.1f}%)")

    return topic_model, df, topics


print("Helper defined: run_sample()")

Helper defined: run_sample()


## Section 3 — Run: English 10 k Sample

In [ ]:
print("=" * 60)
print("Language: en")
print("=" * 60)
en_model, en_df, en_topics = run_sample("en")

## Section 4 — Run: Vietnamese 10 k Sample

In [7]:
print("=" * 60)
print("Language: vi")
print("=" * 60)
vi_model, vi_df, vi_topics = run_sample("vi")

Language: vi
[load_from_duckdb] Loading 116,756 rows …
[load_from_duckdb] docs: 116,756  embeddings: (116756, 768)
Total vi reviews: 116,756
Sampled        : 10,000
Embedding shape: (10000, 768)
Sample doc [0] : 'khách_sạn có sảnh hơi dài đủ để tập_thể_dục trước khi lên phòng có bể_bơi nhưng tôi không có thời_gian thưởng_thức nhân_'


2026-06-09 14:32:30,966 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-09 14:32:40,566 - BERTopic - Dimensionality - Completed ✓
2026-06-09 14:32:40,567 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-09 14:32:40,745 - BERTopic - Cluster - Completed ✓
2026-06-09 14:32:40,746 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-06-09 14:32:40,975 - BERTopic - Representation - Completed ✓
2026-06-09 14:32:40,976 - BERTopic - Topic reduction - Reducing number of topics
2026-06-09 14:32:40,992 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-09 14:33:06,736 - BERTopic - Representation - Completed ✓
2026-06-09 14:33:06,738 - BERTopic - Topic reduction - Reduced number of topics from 82 to 39


Topics  : 38
Outliers: 4,316 / 10,000 (43.2%)


In [8]:
info = vi_model.get_topic_info()
info

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,4316,-1_khách_sạn_phòng_nhân_viên_sạch_sẽ,"[khách_sạn, phòng, nhân_viên, sạch_sẽ, vị_trí,...","[phòng_ốc, khách, đặt phòng, phòng, khách_sạn,...","[phòng, sáng, chúng_tôi, ngon, bữa, bữa sáng, ...","[phòng, nhân_viên, vị_trí, sáng, tốt, thân_thi...",[rose garden residences là nơi tuyệt_vời nhất ...
1,0,2726,0_phòng_sáng_nhân_viên_khách_sạn,"[phòng, sáng, nhân_viên, khách_sạn, bữa, thân_...","[khách_sạn đẹp, thoải_mái, khách, thích, nhân_...","[phòng, sáng, khách_sạn, bữa, bữa sáng, đẹp, n...","[phòng, sáng, nhân_viên, khách_sạn, đẹp, nhiệt...",[kỳ nghỉ tuyệt_vời tại một khách_sạn xinh_xắn ...
2,1,271,1_gần_trung_tâm_nằm_chợ,"[gần, trung_tâm, nằm, chợ, phố, thành_phố, khá...","[thành_phố hồ, bến thành, thành, gần chợ, gần ...","[phố, thành_phố, thuận_tiện, trung_tâm thành_p...","[trung_tâm, nằm, phút, vị_trí, có_thể, cách, đ...",[nó nằm ở trung_tâm thành_phố cách chợ bến thà...
3,2,244,2_quay_dịp_nhiệt_tình_nhiệt_tình quay,"[quay, dịp, nhiệt_tình, nhiệt_tình quay, nhân_...","[chúng_tôi quay, quay trở_lại, quay tiếp, quay...","[quay khách_sạn, quay dịp, thân_thiện, nhân_vi...","[dịp, nhiệt_tình, nhân_viên, thân_thiện, sạch_...","[sẽ quay lại, sẽ quay lại, sẽ quay lại lần sau]"
4,3,220,3_tốt tốt_tốt_tốt tuyệt_vời_tuyệt_vời tốt,"[tốt tốt, tốt, tốt tuyệt_vời, tuyệt_vời tốt, t...","[tốt lắm, tốt đấy, tốt, tốt thật, khá tốt, tuy...","[tốt tốt, tuyệt_vời tuyệt_vời, dịch_vụ tốt, tố...","[tốt, dịch_vụ, lắm, thuong, chất_lượng, đáng, ...","[rất tốt, rất tốt, rất tốt]"
5,4,209,4_giá_giá_cả_rẻ_tiền,"[giá, giá_cả, rẻ, tiền, hợp_lý, giá tiền, giá ...","[tốt giá_cả, giá tốt, tốt giá, giá khá, giá ổn...","[giá, tầm giá, giá_cả hợp_lý, mức giá, giá hợp...","[giá, tiền, tốt, chất_lượng, ổn, mức, phù_hợp,...","[giá ok ổn trong tầm giá, giá rẻ ok tuyệt, giá..."
6,5,209,5_tắm_phòng tắm_nước_phòng,"[tắm, phòng tắm, nước, phòng, toilet, bẩn, khă...","[phòng tắm, tắm phòng, tắm, khăn tắm, tắm thoá...","[tắm, phòng tắm, toilet, bẩn, nước nóng, khăn ...","[tắm, phòng tắm, phòng, toilet, vòi_sen, có_th...",[dụng_cụ tương_đối ổn phòng tắm vòi thiết_kế h...
7,6,207,6_đặt_phòng_đặt phòng_tiền,"[đặt, phòng, đặt phòng, tiền, booking, trả, ch...","[phòng booking, booking khách_sạn, booking, đặ...","[đặt phòng, booking, khách, khách_sạn, thanh_t...","[phòng, trả, thêm, giờ, check, agoda, báo, gọi...",[i came to book a room for 3 days at 6pm today...
8,7,194,7_biển_bãi biển_bãi_gần biển,"[biển, bãi biển, bãi, gần biển, gần, đẹp, nghỉ...","[bờ biển, gần biển, biển đẹp, bờ, sát biển, bã...","[bãi biển, gần biển, khu nghỉ_dưỡng, biển đẹp,...","[biển, bãi, gần, đẹp, nghỉ_dưỡng, view, khách_...","[ok cho đi nhóm gần biển, sạch_sẽ và gần biển,..."
9,8,118,8_thái_độ_lễ_tân_khách_khách_hàng,"[thái_độ, lễ_tân, khách, khách_hàng, ko, tệ, n...","[bảo_vệ thái_độ, thái_độ phục_vụ, hành_lý, thá...","[lễ_tân, khách, dịch_vụ, thiếu, chuyên_nghiệp,...","[thái_độ, nhân_viên, dịch_vụ, ghế, gặp, ngồi, ...",[dịch_vụ rất tệ sảnh lễ_tân rộng nhưng quá nón...


## Section 5 — Topic Tables

In [ ]:
def print_topic_table(model, topics, label: str):
    info = model.get_topic_info()
    info = info[info["Topic"] != -1].sort_values("Count", ascending=False).reset_index(drop=True)

    print(f"\n{'─'*70}")
    print(f"[{label}]  {len(info)} topics")
    print(f"{'topic_id':>8}  {'n_docs':>7}  top_keywords")
    print("─" * 70)
    for _, row in info.iterrows():
        tid     = int(row["Topic"])
        kw_list = model.get_topic(tid)
        kws     = ", ".join(w for w, _ in kw_list[:8]) if kw_list else ""
        print(f"{tid:>8}  {int(row['Count']):>7}  {kws}")

print_topic_table(en_model, en_topics, "en_sample_10k")
print_topic_table(vi_model, vi_topics, "vi_sample_10k")

## Section 6 — Representative Docs for the Largest Topic

In [ ]:
def show_top_topic(model, label: str, n: int = 5):
    info = model.get_topic_info()
    info = info[info["Topic"] != -1].sort_values("Count", ascending=False).reset_index(drop=True)
    top_tid  = int(info.iloc[0]["Topic"])
    top_kws  = ", ".join(w for w, _ in model.get_topic(top_tid)[:8])
    rep_docs = model.get_representative_docs(top_tid)

    print(f"\n=== [{label}] Topic {top_tid}: {top_kws} ===")
    for i, doc in enumerate(rep_docs[:n], 1):
        print(f"\n[{i}] {doc[:300]}")

show_top_topic(en_model, "en_sample_10k")
show_top_topic(vi_model, "vi_sample_10k")

## Section 7 — Inspect Any Topic

In [ ]:
INSPECT_LANG  = "en"   # "en" or "vi"
INSPECT_TOPIC = 0      # topic_id from the table above
N_EXAMPLES    = 5

model  = en_model  if INSPECT_LANG == "en" else vi_model
topics = en_topics if INSPECT_LANG == "en" else vi_topics

kw_list  = model.get_topic(INSPECT_TOPIC)
keywords = ", ".join(w for w, _ in kw_list[:10]) if kw_list else ""
rep_docs = model.get_representative_docs(INSPECT_TOPIC)

info = model.get_topic_info()
row  = info[info["Topic"] == INSPECT_TOPIC]
n_docs = int(row["Count"].values[0]) if len(row) else 0

print(f"[{INSPECT_LANG}_sample_10k]  Topic {INSPECT_TOPIC}  |  {n_docs:,} docs")
print(f"Keywords : {keywords}")
print("-" * 70)
for i, doc in enumerate(rep_docs[:N_EXAMPLES], 1):
    print(f"\n[{i}] {doc[:300]}")